# 08 · Guided MONAI 3D segmentation and transition to real data

    **Audience:** first-year biology students  
    **Time:** about 110 minutes  
    **Prerequisites:** Tutorials 01–07; network access only for first-time dependency installation  
    **Data:** generated locally; no downloads or personal/clinical data.

    ## By the end, you can

- build a MONAI 3D U-Net from paired labeled volumes
- understand transforms, Dice+CE loss and sliding-window inference
- prepare to run the repository's real spleen CT example without confusing synthetic results for clinical validation

    **How to work:** Run cells from top to bottom (Shift+Enter). Inspect every figure, try the **Your turn** prompts, then compare with the supplied self-checks. A GPU is not necessary unless noted. If you restart the runtime, rerun all cells.

## 0 · Setup (run once)
The first code cell installs MONAI in notebook environments that do not have it. This requires internet access for installation, but **no dataset download** is required. On Colab, select *Runtime → Change runtime type*; a CPU is adequate for this small example. If the environment blocks package installation, run the notebook later in Colab.

In [ ]:
import importlib.util, sys, subprocess
if importlib.util.find_spec('monai') is None:
    print('Installing MONAI (requires internet access) ...')
    subprocess.check_call([sys.executable,'-m','pip','install','monai>=1.3,<2'])
import numpy as np
import matplotlib.pyplot as plt
import torch
from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai.transforms import Compose, EnsureChannelFirstd, ScaleIntensityd, EnsureTyped
from monai.data import Dataset, DataLoader

torch.manual_seed(8)
torch.set_num_threads(min(2,torch.get_num_threads()))
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('MONAI imported; device:',DEVICE)

## 1 · Make a miniature, paired volumetric dataset
For learning, we use known synthetic labels. On real projects, 3D segmentation requires **paired images and expert masks**, consistent physical spacing, and split-by-subject evaluation. This synthetic exercise is **not** a clinical or biological benchmark.

In [ ]:
def make_case(seed,size=24):
    rng=np.random.default_rng(seed)
    z,y,x=np.mgrid[:size,:size,:size]
    center=rng.uniform(size*.35,size*.65,3)
    radii=rng.uniform(size*.17,size*.28,3)
    z2=((z-center[0])/radii[0])**2
    y2=((y-center[1])/radii[1])**2
    x2=((x-center[2])/radii[2])**2
    shape=z2+y2+x2
    label=(shape<=1).astype('float32')
    image=np.clip(0.08+0.87*np.exp(-1.5*shape)+rng.normal(0,.07,shape.shape),0,1).astype('float32')
    return {'image':image,'label':label}

training=[make_case(i) for i in range(100,116)]
validation=[make_case(i) for i in range(300,304)]
print('Training/validation cases:',len(training),len(validation))
assert len(training)==16 and len(validation)==4

## 2 · A MONAI dictionary-transform pipeline
Transforms must treat images and masks consistently. `EnsureChannelFirstd` adds the channel axis to `(D,H,W)` arrays. `ScaleIntensityd` normalizes input intensity, **not the binary mask**. `EnsureTyped` converts to tensor-like data. For clinical CT, you would typically use appropriate HU windowing, spacing/orientation corrections and patch-based sampling instead.

In [ ]:
transforms=Compose([
    EnsureChannelFirstd(keys=['image','label'],channel_dim='no_channel'),
    ScaleIntensityd(keys='image'),
    EnsureTyped(keys=['image','label']),
])
train_loader=DataLoader(Dataset(training,transform=transforms),batch_size=2,shuffle=True,num_workers=0)
val_loader=DataLoader(Dataset(validation,transform=transforms),batch_size=1,shuffle=False,num_workers=0)
example=next(iter(train_loader))
print('Batch image:',tuple(example['image'].shape),'Batch mask:',tuple(example['label'].shape))
assert example['image'].shape==example['label'].shape

## 3 · Let MONAI build the network
A U-Net outputs one foreground **logit** at every voxel. `DiceCELoss(sigmoid=True)` applies the sigmoid internally and combines overlap with binary cross-entropy, which helps with foreground/background imbalance.

In [ ]:
model=UNet(spatial_dims=3,in_channels=1,out_channels=1,
           channels=(4,8,16),strides=(2,2),num_res_units=1).to(DEVICE)
loss_fn=DiceCELoss(sigmoid=True)
optimizer=torch.optim.Adam(model.parameters(),lr=0.008)
with torch.no_grad():
    trial=model(example['image'].to(DEVICE))
assert trial.shape==example['label'].shape
print('Network outputs:',tuple(trial.shape))

## 4 · Fit only the small synthetic examples
A real segmentation experiment uses a much larger dataset, patient-level splitting, proper augmentation, spacing and multi-epoch tuning. Here the aim is to observe a working MONAI training loop. The model is **not suitable for real CT**.

In [ ]:
EPOCHS=8
history=[]
for epoch in range(EPOCHS):
    model.train();batch_losses=[]
    for batch in train_loader:
        image=batch['image'].to(DEVICE)
        label=batch['label'].to(DEVICE)
        optimizer.zero_grad()
        logits=model(image)
        loss=loss_fn(logits,label)
        loss.backward();optimizer.step()
        batch_losses.append(float(loss.item()))
    history.append(float(np.mean(batch_losses)))
    if epoch in [0,EPOCHS-1]:
        print(f'epoch {epoch+1}/{EPOCHS}: loss {history[-1]:.3f}')
plt.plot(history);plt.xlabel('Epoch');plt.ylabel('Train Dice+CE loss');plt.show()

## 5 · Sliding-window inference
For large 3D images, a whole volume may not fit in GPU memory. Sliding-window inference predicts overlapping patches then blends them together. Here we use it on a tiny volume to illustrate the API. Apply `sigmoid` to the **returned logits** and threshold at 0.5.

In [ ]:
def dice_score(pred,reference):
    a=pred.bool();b=reference.bool()
    total=int(a.sum()+b.sum())
    return 1. if total==0 else float(2*(a&b).sum()/total)

model.eval()
scores=[]; first=None
with torch.no_grad():
    for batch in val_loader:
        img=batch['image'].to(DEVICE)
        logits=sliding_window_inference(img,roi_size=(16,16,16),sw_batch_size=2,predictor=model,overlap=0.25)
        prob=torch.sigmoid(logits).cpu()
        true=batch['label']
        scores.append(dice_score(prob>0.5,true>0.5))
        if first is None: first=(img.cpu(),true,prob)
print('Held-out Dice:',np.round(scores,3))
print('Mean Dice:',round(float(np.mean(scores)),3))
img,true,prob=first
mid=img.shape[-3]//2
fig,axes=plt.subplots(1,3,figsize=(9,3))
for ax,a,title in zip(axes,[img[0,0,mid],true[0,0,mid],prob[0,0,mid]],['Input','Reference','MONAI probability']):
    ax.imshow(a,cmap='gray',vmin=0,vmax=1);ax.set_title(title);ax.axis('off')
plt.tight_layout();plt.show()

## 6 · From a synthetic exercise to **real CT spleen segmentation**
The repository's existing notebook, [`AdvancedMedicalAI/Volume3DSegmentation/3d_medical_image_segmentation_monai.ipynb`](../AdvancedMedicalAI/Volume3DSegmentation/3d_medical_image_segmentation_monai.ipynb), is the intended next step. It handles NIfTI images, spatial orientation, spacing, CT intensity transforms, foreground-aware patch sampling, and full-volume inference on the Medical Segmentation Decathlon **Task09_Spleen** data.

**Important:** Do **not** apply the model just trained on artificial ellipsoids to patient CT. Its success says nothing about medical performance. In the existing project, use a **patient-level** split; tune using validation only; retain a held-out test set and inspect segmentation errors case by case. The original example needs a larger download and GPU for practical training.

## 7 · Optional: inspect your own paired NIfTI images (not run by default)
If you have appropriately licensed, de-identified and paired `image.nii.gz` and `mask.nii.gz` files, change `USE_REAL_PAIR=True` and provide the paths below. This is **preprocessing inspection only**, not model inference with synthetic weights. Real scans may require orientation/spacing adjustments; inspect metadata carefully.

In [ ]:
USE_REAL_PAIR=False
IMAGE_PATH='image.nii.gz'  # replace only if you have authorized de-identified data
MASK_PATH='mask.nii.gz'
if USE_REAL_PAIR:
    from pathlib import Path
    from monai.transforms import LoadImaged
    if not (Path(IMAGE_PATH).is_file() and Path(MASK_PATH).is_file()):
        raise FileNotFoundError('Set IMAGE_PATH and MASK_PATH to existing NIfTI files.')
    load_pair=Compose([LoadImaged(keys=['image','label']),
                       EnsureChannelFirstd(keys=['image','label']),
                       EnsureTyped(keys=['image','label'])])
    pair=load_pair({'image':IMAGE_PATH,'label':MASK_PATH})
    print('Image and mask shapes:',tuple(pair['image'].shape),tuple(pair['label'].shape))
    assert pair['image'].shape[1:]==pair['label'].shape[1:], 'Check spatial alignment before training.'
else:
    print('Optional real-data inspection skipped: no files required.')

## Your turn
1. Reduce the number of training examples to 4, restart training from a new model and compare validation Dice.
2. Explain why `ScaleIntensityd` should be applied to the **image** and not to the integer label.
3. Name three items that change when moving to spleen CT: image intensity units, voxel spacing/orientation, and memory-efficient training patches.
4. In the original spleen notebook, locate `RandCropByPosNegLabeld` and `sliding_window_inference`; explain why both are useful.

**Completion standard:** You can articulate *data → transforms → U-Net → loss → optimization → validation → inference*, and you can explain why a synthetic toy model cannot be clinically deployed.

In [ ]:
assert len(scores)==len(validation)
assert all(np.isfinite(scores))
assert 0<=float(np.mean(scores))<=1
print('Self-checks passed: training, validation and inference completed.')